In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import pickle as pkl
import ast
import argparse

from Model.MENDR.SpatialTemporalEncoder import SpatialTemporalEncoder
from Model.MENDR.MENDRContextualizer import mATTContextualizer
from Model.MENDR.MENDRTrainer import MENDRTrainer
from Model.encoder import ConvEncoder
from Model.MENDR.R2E import R2E
from Model.transforms import RandomTemporalCrop
from Model.TrainingDecoder.trainingDecoder import ConvDecoder

In [3]:
# Open the file in read binary mode
with open('/home/hice1/mchen439/scratch/models/model_4eabfb7fb3e84cfda13ea1e503a8261d.pkl', 'rb') as file:
    # Load the pickled data from the file
    model = pkl.load(file)

In [4]:
# setup arg parser
parser = argparse.ArgumentParser()

# Training Parameters
parser.add_argument(
    "--random_state", type=int, help="Random state for reproducibility", default=42
)
parser.add_argument(
    "-b", "--batch_size", default=512, type=int, help="mini-batch size (default: 512)"
)

parser.add_argument(
    "-e", "--training_epochs", default=100, type=int, help="number of total training epochs (default: 100)"
)

parser.add_argument(
    "--train_frac", type=float, help="Fraction of dataset to use for training", default=1.0
)

parser.add_argument(
    "--val_frac", type=float, help="Fraction of dataset to use for validation", default=0.0
)


# Spatial Temporal Embedding Encoder Configs
parser.add_argument(
    "--seq_len", type=int, help="Sequence Length of Recordings", default=15360
)

parser.add_argument(
    "--pred_len", type=int, help="Prediction Length of Recordings, should be equal to sequence length",  default=15360
)

parser.add_argument(
    "--top_k", type=int, help="Top K frequencies to select in TimesBlock", default=5
)

parser.add_argument(
    "--d_model", type=int, help="Number of EEG Channels", default=19
)

parser.add_argument(
    "--d_ff", type=int, help="Feed Forward Dimension in TimesBlock", default=9
)

parser.add_argument(
    "--num_kernels", type=int, help="Number of Kernels in Convolutional Layer of Inception Block in TimesBlock", default=3
)

parser.add_argument(
    "--num_heads", type=int, help="Number of Heads in Multihead Attention Layer of GAT",  default=4
)

parser.add_argument(
    "--ste_dropout", type=float, help="Dropout Rate for GAT", default=0.1
)

# Encoder Configs
parser.add_argument(
    "--encoder_h", type=int, help="Hidden Dimension of Encoder", default=256
)

parser.add_argument(
    "--enc_width", type=ast.literal_eval, help="Encoder Width", default=(2, 2)
)

parser.add_argument(
    "--enc_downsample", type=ast.literal_eval, help="Encoder Downsample", default=(2, 2)
)

parser.add_argument(
    "--enc_dropout", type=float, help="Dropout Rate for Encoder", default=0.1
)

# MENDR Contextualizer Configs
parser.add_argument(
    "--in_features", type=int, help="Input Features for Contextualizer", default=256
)

parser.add_argument(
    "--dropout", type=float, help="Dropout Rate for Contextualizer", default=0.1
)

parser.add_argument(
    "--position_encoder", type=int, help="Position Encoder", default=25
)

parser.add_argument(
    "--epochs", type=int, help="Number of Epochs for mATT", default=1
)

# Trainer Configs
parser.add_argument(
    "--mask_span", type=int, help="Mask Span", default=6
)

parser.add_argument(
    "--mask_rate", type=float, help="Mask Rate", default=0.1
)

parser.add_argument(
    "--multi_gpu", type=bool, help="Multi GPU Training", default=False
)

parser.add_argument(
    "--encoder_grad_frac", type=float, help="Encoder Gradient Fraction", default=1
)

parser.add_argument(
    "--learning_rate", type=float, help="Learning Rate", default=1e-3
)

parser.add_argument(
    "--l2_weight_decay", type=float, help="L2 Weight Decay. Helps with generalization", default=1e-5
)

parser.add_argument(
    "--temp", type=float, help="Temperature", default=0.1
)

parser.add_argument(
    "--enc_feat_l2", type=float, help="Encoder Feature L2", default=1e-5
)

parser.add_argument(
    "--num_negatives", type=int, help="Number of Negatives in Contrastive Learning Task", default=10
)

parser.add_argument(
    "--max_crop_frac", type=float, help="Maximum Crop Fraction on Random Temporal Crop", default=0.05
)

parser.add_argument(
    "--permuted_encodings", type=bool, help="Permuted Encodings", default=False
)

parser.add_argument(
    "--permuted_contexts", type=bool, help="Permuted Contexts", default=False
)

parser.add_argument(
    "--save_model", type=bool,
    help="Whether to save model.", default=False
)

parser.add_argument(
    "--save_model_directory", type=str, help="If save model is true, specifies where to save it", required=False
)

# parse args
args, unknown_args = parser.parse_known_args()

In [5]:
### Model ###
stEncoder = SpatialTemporalEncoder(args)
encoder = ConvEncoder(in_features=args.d_model, encoder_h=args.encoder_h, 
                    enc_width=args.enc_width, dropout=args.enc_dropout, enc_downsample=args.enc_downsample)
contextualizer = mATTContextualizer(args)
r2e = R2E(args)

decoder = ConvDecoder(encoder_h=args.encoder_h,
                    out_features=args.d_model, 
                    dec_width=args.enc_width[::-1],
                    dropout=args.enc_dropout,
                    dec_upsample=args.enc_downsample[::-1],
                    original_time_len=args.seq_len,
                    top_k=args.top_k,
                    num_kernels=args.num_kernels,
                    device='cuda' if torch.cuda.is_available() else 'cpu')

trainer = MENDRTrainer(stEncoder, encoder, contextualizer, r2e, decoder, args)
print("Total number of parameters: ", sum(p.numel() for p in trainer.parameters() if p.requires_grad))

trainer.set_optimizer(torch.optim.Adam(trainer.parameters()))
trainer.add_batch_transform(RandomTemporalCrop(max_crop_frac=args.max_crop_frac))

GPU(s) detected: training and model execution will be performed on GPU.
Device Properties: _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81157MB, multi_processor_count=108)
Total number of parameters:  2832533478


In [6]:
trainer.load_best(model)

contextualizer


RuntimeError: Error(s) in loading state_dict for mATTContextualizer:
	Missing key(s) in state_dict: "mask_replacement", "relative_position.0.bias", "relative_position.0.parametrizations.weight.original0", "relative_position.0.parametrizations.weight.original1", "input_conditioning.1.weight", "input_conditioning.1.bias", "input_conditioning.4.weight", "input_conditioning.4.bias", "att.q_trans.weight", "att.k_trans.weight", "att.v_trans.weight", "ract2.epsilon". 
	Unexpected key(s) in state_dict: "tangent.vec.row_idx", "tangent.vec.col_idx". 

In [23]:
print(len(model), type(model[0]))

6 <class 'dict'>


In [24]:
for module in model:
    for k, v in module.items():
        print(k)

tangent.vec.row_idx
tangent.vec.col_idx
times_block.conv.0.kernels.0.weight
times_block.conv.0.kernels.0.bias
times_block.conv.0.kernels.1.weight
times_block.conv.0.kernels.1.bias
times_block.conv.0.kernels.2.weight
times_block.conv.0.kernels.2.bias
times_block.conv.2.kernels.0.weight
times_block.conv.2.kernels.0.bias
times_block.conv.2.kernels.1.weight
times_block.conv.2.kernels.1.bias
times_block.conv.2.kernels.2.weight
times_block.conv.2.kernels.2.bias
gnn_layer.conv.att_src
gnn_layer.conv.att_dst
gnn_layer.conv.bias
gnn_layer.conv.lin.weight
gnn_layer.projection.weight
gnn_layer.projection.bias
predict_linear.weight
predict_linear.bias
project_back.weight
project_back.bias
mask_replacement
relative_position.0.bias
relative_position.0.parametrizations.weight.original0
relative_position.0.parametrizations.weight.original1
input_conditioning.1.weight
input_conditioning.1.bias
input_conditioning.4.weight
input_conditioning.4.bias
att.q_trans.weight
att.k_trans.weight
att.v_trans.weight